In [0]:
# =============================================================================
# Notebook: 07_reconciliation
# Purpose : Verify data integrity between Bronze (raw/source) and Silver
#           (cleaned/target) layers after the CDC pipeline runs. Confirms no
#           data was unexpectedly lost, duplicated, or corrupted during
#           transformation.
# =============================================================================

from pyspark.sql import functions as F

storage_account = "stretailcdcproj"
bronze_path = f"abfss://bronze@{storage_account}.dfs.core.windows.net/retail_orders/"
silver_path = f"abfss://silver@{storage_account}.dfs.core.windows.net/retail_orders/"

df_bronze = spark.read.format("delta").load(bronze_path)
df_silver = spark.read.format("delta").load(silver_path)

print("=" * 60)
print("ETL RECONCILIATION REPORT: Bronze vs Silver")
print("=" * 60)

# ---- Check 1: Row Count Reconciliation ----
# Silver should have fewer or equal rows than Bronze, since duplicates are
# removed during the Bronze -> Silver transformation. We calculate the
# EXPECTED Silver count (distinct order_id in Bronze) and compare it to the
# ACTUAL Silver row count - they must match exactly.
bronze_count = df_bronze.count()
silver_count = df_silver.count()
expected_silver_count = df_bronze.select("order_id").distinct().count()

print(f"\n--- Row Count Check ---")
print(f"Bronze total rows          : {bronze_count}")
print(f"Bronze distinct order_ids  : {expected_silver_count}")
print(f"Silver actual rows         : {silver_count}")

if silver_count == expected_silver_count:
    print("PASSED: Silver row count matches expected distinct order_id count")
else:
    diff = silver_count - expected_silver_count
    print(f"FAILED: Mismatch of {diff} rows — investigate immediately")

# ---- Check 2: Sum Reconciliation (financial control total) ----
# compare aggregate totals (like sum of amount) between source and target. 
# Since Silver keeps only the LATEST version per order_id, we compute the "expected" 
# sum using the same latest - record-per-order_id logic on Bronze, then compare to Silver's actual sum.
from pyspark.sql import Window

window_spec = Window.partitionBy("order_id").orderBy(F.col("last_modified").desc())
df_bronze_latest = (
    df_bronze
    .withColumn("row_num", F.row_number().over(window_spec))
    .filter(F.col("row_num") == 1)
    .drop("row_num")
)

expected_sum = df_bronze_latest.agg(F.sum("amount")).collect()[0][0]
actual_sum = df_silver.agg(F.sum("amount")).collect()[0][0]

print(f"\n--- Sum Reconciliation (amount) ---")
print(f"Expected total amount (from Bronze, latest per order_id): {expected_sum:,.2f}")
print(f"Actual total amount (Silver)                            : {actual_sum:,.2f}")

# Allow tiny floating point tolerance instead of exact equality
tolerance = 0.01
if abs(expected_sum - actual_sum) < tolerance:
    print("PASSED: Sum totals match within tolerance")
else:
    print(f"FAILED: Sum mismatch of {abs(expected_sum - actual_sum):,.2f}")

# ---- Check 3: Referential Check ----
# Every order_id present in Silver must exist somewhere in Bronze - Silver
# should never contain "phantom" records that don't trace back to source.
silver_ids = df_silver.select("order_id").distinct()
bronze_ids = df_bronze.select("order_id").distinct()

orphan_records = silver_ids.subtract(bronze_ids).count()

print(f"\n--- Referential Integrity Check ---")
print(f"Silver order_ids not found in Bronze: {orphan_records}")

if orphan_records == 0:
    print("PASSED: Every Silver record traces back to a valid Bronze record")
else:
    print(f"FAILED: {orphan_records} orphan records found in Silver")

# ---- Check 4: Row-level Hash Comparison (spot-check data accuracy) ----
# Beyond counts and sums, we verify actual field-level values weren't
# corrupted during transformation, using a hash of key business columns.
df_bronze_hash = df_bronze_latest.withColumn(
    "row_hash",
    F.sha2(F.concat_ws("|", "order_id", "customer_id", "product", "region", "quantity", "amount"), 256)
).select("order_id", "row_hash")

df_silver_hash = df_silver.withColumn(
    "row_hash",
    F.sha2(F.concat_ws("|", "order_id", "customer_id", "product", "region", "quantity", "amount"), 256)
).select("order_id", "row_hash")

mismatched = (
    df_bronze_hash.alias("b")
    .join(df_silver_hash.alias("s"), on="order_id", how="inner")
    .filter(F.col("b.row_hash") != F.col("s.row_hash"))
    .count()
)

print(f"\n--- Row-level Hash Comparison ---")
print(f"Records with mismatched field values: {mismatched}")

if mismatched == 0:
    print("PASSED: All matched records have identical field values")
else:
    print(f"FAILED: {mismatched} records have data value mismatches")

# ---- Final Summary ----
print("\n" + "=" * 60)
print("RECONCILIATION SUMMARY")
print("=" * 60)
checks_passed = (
    (silver_count == expected_silver_count) +
    (abs(expected_sum - actual_sum) < tolerance) +
    (orphan_records == 0) +
    (mismatched == 0)
)
print(f"Checks passed: {checks_passed} / 4")
if checks_passed == 4:
    print("ALL RECONCILIATION CHECKS PASSED — data is trustworthy")
else:
    print("One or more checks FAILED — review before promoting to Gold layer")

ETL RECONCILIATION REPORT: Bronze vs Silver

--- Row Count Check ---
Bronze total rows          : 5170
Bronze distinct order_ids  : 5100
Silver actual rows         : 5100
PASSED: Silver row count matches expected distinct order_id count

--- Sum Reconciliation (amount) ---
Expected total amount (from Bronze, latest per order_id): 1,281,514.01
Actual total amount (Silver)                            : 1,281,514.01
PASSED: Sum totals match within tolerance

--- Referential Integrity Check ---
Silver order_ids not found in Bronze: 0
PASSED: Every Silver record traces back to a valid Bronze record

--- Row-level Hash Comparison ---
Records with mismatched field values: 0
PASSED: All matched records have identical field values

RECONCILIATION SUMMARY
Checks passed: 4 / 4
ALL RECONCILIATION CHECKS PASSED — data is trustworthy
